# Qwen3-4B Multi-Agent Test Environment

## Setup
- Cell 1: pip install + 코드 전체 + 모델 로드 (최초 1회)
- Cell 2: 에이전트 생성 + 프롬프트 정의
- Cell 3+: 실험 (복사해서 쓰면 됨)

---

## API Reference

### Agent 생성
```python
agent = Agent("이름", "system prompt")
```

### 단일 추론
```python
# 기본 (thinking OFF, 256 tokens)
r = agent.say("질문")

# thinking ON + 토큰 늘리기
r = agent.say("질문", max_tokens=2048, thinking=True)

# 결과 접근
r["response"]   # 응답 텍스트
r["tokens"]     # 생성 토큰 수
r["time"]       # 소요 시간(초)
```

### 파라미터
| 파라미터 | 기본값 | 설명 |
|----------|--------|------|
| `max_tokens` | 256 | 생성 최대 토큰. thinking=True면 2048 권장 |
| `thinking` | False | True: Qwen3 내부 추론(`<think>`) 활성화. 느리지만 정확도↑ |

### 프롬프트 변경
```python
agent.set_prompt("새 system prompt")
```

### A → B 통신
```python
result = send(a, b, "메시지")
result["sender"]["response"]    # A의 응답
result["receiver"]["response"]  # B의 응답
result["total_tokens"]          # 총 토큰
```

### A → B → C 체인
```python
result = chain([a, b, c], "메시지")
result["final"]          # 마지막 에이전트 응답
result["total_tokens"]   # 총 토큰
result["steps"]          # 각 단계별 결과 리스트
```

### ask() 헬퍼 (ABCD 문제용)
```python
r = ask(agent, MATH_PROMPT, "질문텍스트", max_tokens=512)
r["answer"]     # 추출된 답 (A/B/C/D 또는 N/A)
r["response"]   # 전체 응답
r["tokens"]     # 토큰 수
r["time"]       # 소요 시간
```

### 유틸리티
```python
extract_number("답은 42입니다")   # → 42.0
grade(got, expected)              # 채점 (10% 오차 허용)
```

In [ ]:
# ============================================================
# Cell 1: 모델 로드 (최초 1회)
# ============================================================
!pip install -q torch transformers accelerate

import torch
import time
import re

# Global model/tokenizer
_model = None
_tokenizer = None
_device = None

def load_model(model_id: str = "Qwen/Qwen3-4B"):
    global _model, _tokenizer, _device
    from transformers import AutoModelForCausalLM, AutoTokenizer
    _device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if _device == "cuda" else torch.float32
    print(f"Device: {_device}")
    if _device == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Loading {model_id}...")
    t0 = time.time()
    _tokenizer = AutoTokenizer.from_pretrained(model_id)
    _model = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=dtype,
        device_map="auto" if _device == "cuda" else None,
    )
    if _device == "cpu":
        _model = _model.to(_device)
    params = sum(p.numel() for p in _model.parameters()) / 1e9
    print(f"Loaded in {time.time()-t0:.1f}s ({params:.1f}B params)")

class Agent:
    def __init__(self, name: str, system_prompt: str):
        self.name = name
        self.system_prompt = system_prompt
        self.history = []

    def say(self, message: str, max_tokens: int = 256, thinking: bool = False) -> dict:
        if _model is None:
            raise RuntimeError("load_model()을 먼저 실행하세요.")
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": message},
        ]
        text = _tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=thinking
        )
        inputs = _tokenizer(text, return_tensors="pt").to(_model.device)
        t0 = time.time()
        with torch.no_grad():
            output = _model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
        elapsed = time.time() - t0
        response = _tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        gen_tokens = output.shape[1] - inputs["input_ids"].shape[1]
        result = {"response": response, "tokens": gen_tokens, "time": round(elapsed, 1)}
        self.history.append({"input": message, **result})
        return result

    def set_prompt(self, new_prompt: str):
        self.system_prompt = new_prompt
        return self

    def __repr__(self):
        return f"Agent('{self.name}')"

def send(sender, receiver, message, max_tokens=256, verbose=True):
    s = sender.say(message, max_tokens=max_tokens)
    r = receiver.say(s["response"], max_tokens=max_tokens)
    if verbose:
        print(f"[{sender.name}] {s['tokens']}tok, {s['time']}s")
        print(f"  >> {s['response'][:200]}")
        print(f"[{receiver.name}] {r['tokens']}tok, {r['time']}s")
        print(f"  >> {r['response'][:200]}")
    return {"sender": s, "receiver": r, "tx_tokens": s["tokens"], "total_tokens": s["tokens"] + r["tokens"]}

def chain(agents, message, max_tokens=256, verbose=True):
    current = message
    results = []
    for agent in agents:
        r = agent.say(current, max_tokens=max_tokens)
        results.append({"agent": agent.name, **r})
        if verbose:
            print(f"[{agent.name}] {r['tokens']}tok, {r['time']}s")
            print(f"  >> {r['response'][:200]}")
        current = r["response"]
    return {"steps": results, "final": results[-1]["response"], "total_tokens": sum(r["tokens"] for r in results)}

def extract_number(text):
    nums = re.findall(r'-?[\d,]+\.?\d*', text.replace(',', ''))
    return float(nums[0]) if nums else -999

def grade(got, expected, tolerance=0.1):
    if expected == 0:
        return abs(got) < tolerance
    return abs(got - expected) / abs(expected) < tolerance

load_model("Qwen/Qwen3-4B")

In [ ]:
# ============================================================
# Cell 2: 에이전트 생성 (Math / Science)
# ============================================================
SYSTEM_PROMPT = "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."

MATH_PROMPT = (
    "You are a math agent. Given the input question, reason step-by-step "
    "and put the final answer inside \\boxed{YOUR_FINAL_ANSWER}. "
    "Your final answer must be selected from A,B,C,D. "
    "For example \\boxed{A}. Do not add any other contents inside the box."
)

SCIENCE_PROMPT = (
    "You are a science agent. Given the input question, reason step-by-step "
    "and put the final answer inside \\boxed{YOUR_FINAL_ANSWER}. "
    "Your final answer must be selected from A,B,C,D. "
    "For example \\boxed{A}. Do not add any other contents inside the box."
)

math_agent = Agent("MathAgent", SYSTEM_PROMPT)
science_agent = Agent("ScienceAgent", SYSTEM_PROMPT)

def ask(agent, prompt_template, question, max_tokens=512):
    """프롬프트 템플릿 + 질문으로 추론 실행."""
    full_prompt = f"{prompt_template}\nQuestion: {question}\nYour response:"
    r = agent.say(full_prompt, max_tokens=max_tokens)
    import re
    match = re.search(r'\\boxed\{([A-D])\}', r["response"])
    answer = match.group(1) if match else "N/A"
    print(f"[{agent.name}] {r['tokens']}tok, {r['time']}s")
    print(f"  Answer: {answer}")
    print(f"  Response: {r['response'][:300]}")
    return {"answer": answer, **r}

print(math_agent, science_agent)

In [ ]:
# ---- Math Agent: 단일 문제 ----
q1 = """If f(x) = 2x^2 - 3x + 1, what is f(3)?
A) 10
B) 12
C) 14
D) 16"""

r1 = ask(math_agent, MATH_PROMPT, q1)

In [ ]:
# ---- Math Agent: 확률 문제 ----
q2 = """A bag contains 3 red and 5 blue balls. Two balls are drawn without replacement.
What is the probability that both are red?
A) 3/28
B) 3/8
C) 9/64
D) 1/7"""

r2 = ask(math_agent, MATH_PROMPT, q2)

In [ ]:
# ---- Science Agent: 물리 문제 ----
q3 = """An object is dropped from rest. How far does it fall in the first 3 seconds?
(Use g = 10 m/s^2)
A) 15 m
B) 30 m
C) 45 m
D) 90 m"""

r3 = ask(science_agent, SCIENCE_PROMPT, q3)

In [ ]:
# ---- Science Agent: 화학 문제 ----
q4 = """What is the molecular formula of glucose?
A) C6H10O5
B) C6H12O6
C) C12H22O11
D) CH3COOH"""

r4 = ask(science_agent, SCIENCE_PROMPT, q4)

In [ ]:
# ---- 결과 요약 ----
results = [
    ("Math - f(3)", r1["answer"], "A"),       # 2(9)-3(3)+1 = 10
    ("Math - Prob", r2["answer"], "A"),       # 3/8 * 2/7 = 3/28
    ("Science - Physics", r3["answer"], "C"), # 0.5*10*9 = 45m
    ("Science - Chem", r4["answer"], "B"),    # C6H12O6
]

print("=" * 40)
print(f"{'Problem':<22} {'Got':<5} {'Exp':<5} {'Result'}")
print("-" * 40)
correct = 0
for name, got, expected in results:
    ok = got == expected
    correct += ok
    print(f"{name:<22} {got:<5} {expected:<5} {'OK' if ok else 'FAIL'}")
print("-" * 40)
print(f"Accuracy: {correct}/{len(results)} ({correct/len(results)*100:.0f}%)")